This is a kRPC Tutorial

In [2]:
import krpc
import csv
import time

conn = krpc.connect(name='My First Script')
vessel = conn.space_center.active_vessel

print(vessel.name)
print(vessel.met)          # mission elapsed time, in seconds
print(vessel.mass)         # current mass, in kg — changes as you burn fuel!

Twin Boar 2
0.0
622825.0


2. Two things worth noticing immediately, because they'll bite you later if you don't: vessel.flight() is a method call, with parentheses — vessel.orbit is not. There's no way to guess which is which; check Appendix A rather than assume. And flight()'s default reference frame isn't always what you want — fine for altitude and surface speed, but Lesson 9 comes back to this gap for real once it matters for a landing.

Try it: print flight.mean_altitude, flight.speed, orbit.apoapsis_altitude, and orbit.periapsis_altitude once on the pad and once in orbit. The pad reading should make apoapsis/periapsis look strange (near zero or negative) — correct, not a bug: sitting on the surface doesn't have a real orbit yet.

In [ ]:
flight = vessel.flight()
orbit = vessel.orbit

print(flight.mean_altitude)       # meters above sea level
print(flight.surface_altitude)    # meters above terrain directly below
print(flight.speed)               # m/s, relative to the surface
print(orbit.apoapsis_altitude)    # meters
print(orbit.periapsis_altitude)   # meters
print(orbit.period)               # seconds — compare this to your Phase 02 prediction

3. The Problem with Asking One at a Time: Streams
Every single property read is a round trip to the game. Set up a stream once instead of paying that cost every loop tick.

import time
while True:
    print(vessel.flight().mean_altitude)   # slow: a full round trip every call
    time.sleep(0.05)

Do that fast enough or for enough values at once, and the round-trip delay itself starts to matter — you can end up acting on altitude that's slightly stale. kRPC's fix is a stream: set it up once, and the server pushes updates without a fresh round trip per read.

conn.add_stream(getattr, obj, 'property_name') is the general pattern for streaming any property. Call the stream like a function to get its current value — cheaply, since the expensive round trip already happened once.

In [ ]:
altitude = conn.add_stream(getattr, vessel.flight(), 'mean_altitude')

while altitude() < 1000:
    print(altitude())
    time.sleep(0.1)

altitude.remove()   # clean up when you're done with it

4. Writing Values to a File
A print statement disappears the moment your terminal scrolls past it. Save the numbers instead.

Try it: extend the script to also log vessel.thrust and vessel.available_thrust, and compute a live TWR each row (thrust / (mass * 9.81)). Compare a few rows against KER's on-screen readout from the same flight — Design Challenge 1's "cross-check your hand TWR against KER" step, done by script instead of by eye.


In [11]:
import csv
import time

conn = krpc.connect(name='Mini Logger')
vessel = conn.space_center.active_vessel

altitude = conn.add_stream(getattr, vessel.flight(), 'mean_altitude')
speed = conn.add_stream(getattr, vessel.flight(), 'speed')
mass = conn.add_stream(getattr, vessel, 'mass')

with open('mini_log.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['time', 'altitude', 'speed', 'mass'])
    start = time.time()
    while altitude() < 10000:      # stop logging once we clear 10 km
        writer.writerow([time.time() - start, altitude(), speed(), mass()])
        time.sleep(0.2)
altitude.remove()
speed.remove()
mass.remove()

5.  Sending Commands: Throttle, Staging & Control Inputs
Everything so far has been read-only. Commands go through vessel.control — and now a mistake can actually hurt your rocket.

Be careful here in a way you didn't need to be in Lessons 1–4. A read can't hurt your rocket. A command can — setting throttle to 1.0 at the wrong moment, or staging early, has the exact same consequences it would if you'd pressed the key yourself. Test control-issuing scripts on a cheap, disposable craft before trusting them on a design you care about.

Try it: write a script that sets throttle to 1.0, waits until flight.mean_altitude passes 500 m, then cuts throttle to 0. Time it against a hand-flown attempt at the same thing — a tiny, safe first taste of the automation Design Challenge 3 asks for. It's also the crudest possible version of a landing burn (full throttle, hard cutoff), which is exactly why Lesson 10 exists later.

In [3]:
control = vessel.control

control.throttle = 1.0          # full throttle, 0.0 to 1.0
control.activate_next_stage()   # trigger the next staging event
control.sas = True              # turn on Stability Assist
control.rcs = True               # turn on RCS thrusters

6.  Pointing the Ship: auto_pilot
Staging and throttle control when the engine fires. auto_pilot controls which way the ship is pointed.

target_pitch_and_heading(pitch, heading) takes pitch in degrees above the horizon (90 = straight up) and heading in compass degrees (90 = due east). ap.wait() is a blocking call — your script pauses until the autopilot reports it's actually settled, so you don't start a burn while still rotating.

Try it: script a simple gravity turn — pitch from 90° to 45° linearly between 1,000 m and 10,000 m of altitude, recomputing the target pitch every loop iteration from a stream rather than repeated property reads.  

A scripted gravity turn that recomputes its target pitch from a stream, and settles before ap.wait() returns.


In [22]:
control = vessel.control

control.throttle = 1.0          # full throttle, 0.0 to 1.0
control.activate_next_stage()   # trigger the next staging event
control.sas = True              # turn on Stability Assist
control.rcs = True               # turn on RCS thrusters

# neophyte code
time.sleep(30)                    # wait 30 seconds

#max_angular_velocity = (0.00001, 0.00001, 0.00001)  # rad/s
ap = vessel.auto_pilot

ap.auto_tune = False
ap.pitch_pid_gains=(1, 0.2, 0.1)
ap.roll_pid_gains=(1, 0.2, 0.1)
ap.yaw_pid_gains=(1, 0.2, 0.1)

# This is now a flag instead of a call
ap.engaged = True
ap.target_pitch_and_heading(0, 90)   # straight up, facing east
#print(max_angular_velocity)
ap.wait()                              # blocks until the ship is actually pointed there


# ... later, after your burn ...
ap.engaged = False

Checking Your Own Physics: a Vis-Viva Live-Checker
A script that doesn't fly anything — it watches energy conservation hold, live, the whole way around an unpowered ellipse.

The idea
Phase 03's vis-viva equation predicts speed anywhere on an orbit from just three numbers: v² = μ·(2/r − 1/a). Every script so far has read a number once and moved on; this one reads two numbers every loop tick, feeds them into a function, and checks the result against a third number kRPC measures directly — a genuinely different use of a loop than staging or a burn cutoff.

Run this coasting on an unpowered ellipse — after one of Design Challenge 4's apoapsis-raising burns, engine off — and the predicted-vs-measured gap should stay small and roughly constant everywhere on the orbit, near periapsis and near apoapsis alike. vessel.orbit.semi_major_axis and vessel.orbit.radius are both live properties: a barely moves on a stable coast, r swings the whole way from periapsis to apoapsis and back, and vis-viva ties the two together into the speed you're actually measuring.

In [ ]:
import krpc, time, math

conn = krpc.connect(name='Vis-Viva Checker')
vessel = conn.space_center.active_vessel
body = vessel.orbit.body

def vis_viva_speed(mu, r, a):
    return math.sqrt(mu * (2 / r - 1 / a))

mu = body.gravitational_parameter
flight = vessel.flight(vessel.orbital_reference_frame)

while True:
    a = vessel.orbit.semi_major_axis
    r = vessel.orbit.radius       # current distance from the body's center
    predicted = vis_viva_speed(mu, r, a)
    measured = flight.speed
    diff = predicted - measured
    print(f"r={r:,.0f}  predicted={predicted:,.1f}  measured={measured:,.1f}  diff={diff:,.2f}")
    time.sleep(1)

Maneuver Nodes: the Class That Runs the Duna Transfer
A maneuver node is created, pointed, sized, and then executed. This is the skill Design Challenge 5's ejection burn asks for.

Notice what's genuinely new compared to Lessons 5–6: you're not commanding the ship directly, you're creating a plan (the node), pointing the ship at the plan's own reference frame, and burning until the plan reports it's satisfied — reading remaining_delta_v the same way you'd read altitude, as a live number that counts down. This is precisely why the main manual insists on hand-verifying MechJeb's or your own node's phase angle against your own Kepler-based calculation: the node executes exactly what you told it to, correct or not.

Try it: create a node for a small prograde burn (50 m/s) 60 seconds in the future, print node.time_to, node.remaining_delta_v, and node.orbit.apoapsis_altitude once, then remove the node without executing it. Get comfortable creating and inspecting nodes before executing one for real.

In [ ]:
node = vessel.control.add_node(
    conn.space_center.ut + 300,   # universal time: 300 seconds from now
    prograde=850                  # m/s of prograde Δv
)

remaining = conn.add_stream(getattr, node, 'remaining_delta_v')

vessel.auto_pilot.engage()
vessel.auto_pilot.reference_frame = node.reference_frame
vessel.auto_pilot.target_direction = (0, 1, 0)   # node's own "burn direction" axis
vessel.auto_pilot.wait()

vessel.control.throttle = 1.0
while remaining() > 5:     # cut throttle when within 5 m/s of the target
    pass
vessel.control.throttle = 0.0

node.remove()